# **Regras**

## Base de risco crédito

In [1]:
import Orange

In [2]:
base_risco_credito = Orange.data.Table('../data/risco_credito_regras.csv')
base_risco_credito

[[ruim, alta, nenhuma, 0_15 | alto],
 [desconhecida, alta, nenhuma, 15_35 | alto],
 [desconhecida, baixa, nenhuma, 15_35 | moderado],
 [desconhecida, baixa, nenhuma, acima_35 | alto],
 [desconhecida, baixa, nenhuma, acima_35 | baixo],
 ...
]

In [3]:
base_risco_credito.domain

[historia, divida, garantias, renda | risco]

In [4]:
cn2 = Orange.classification.rules.CN2Learner()
regras_risco_credito = cn2(base_risco_credito)

In [5]:
for regras in regras_risco_credito.rule_list:
    print(regras)

IF renda==0_15 THEN risco=alto 
IF historia==boa AND divida!=alta THEN risco=baixo 
IF historia==boa AND garantias!=nenhuma THEN risco=baixo 
IF historia==boa AND renda!=15_35 THEN risco=baixo 
IF historia==boa THEN risco=moderado 
IF divida==alta THEN risco=alto 
IF historia!=desconhecida THEN risco=moderado 
IF garantias==adequada THEN risco=baixo 
IF renda==15_35 THEN risco=moderado 
IF historia==desconhecida THEN risco=baixo 
IF TRUE THEN risco=alto 


In [6]:
previsoes = regras_risco_credito([['boa', 'alta', 'nenhuma', 'acima_35'], ['ruim', 'alta', 'adequada', '0_15']])

In [7]:
previsoes

array([1, 0])

In [8]:
base_risco_credito.domain.class_var.values

('alto', 'baixo', 'moderado')

In [9]:
for i in previsoes:
    print(base_risco_credito.domain.class_var.values[i])

baixo
alto


## Base credit data

In [10]:
base_credit = Orange.data.Table('../data/credit_data_regras.csv')
base_credit

[[66155.9, 59.017, 8106.53 | 0],
 [34415.2, 48.1172, 6564.75 | 0],
 [57317.2, 63.108, 8020.95 | 0],
 [42709.5, 45.752, 6103.64 | 0],
 [66952.7, 18.5843, 8770.1 | 1],
 ...
]

In [11]:
base_credit.domain

[income, age, loan | default]

In [12]:
base_dividida = Orange.evaluation.testing.sample(base_credit, n=0.25)
base_dividida

([[27334.6, 42.6712, 2963.79 | 0],
  [57457.9, 50.7147, 3608.81 | 0],
  [39546, 43.7009, 5787.66 | 0],
  [57018.5, 44.8257, 3507.25 | 0],
  [69939.3, 55.6376, 2225.22 | 0],
  ...
 ],
 [[65705, 50.9284, 1969.79 | 0],
  [40453.9, 20.7099, 890.94 | 0],
  [51915.7, 44.1091, 2282.91 | 0],
  [51790.7, 41.1509, 1281.04 | 0],
  [63062, 39.2016, 1850.37 | 0],
  ...
 ])

In [13]:
base_train = base_dividida[1]
base_test = base_dividida[0]
len(base_train), len(base_test)

(1500, 500)

In [14]:
regras_credit = cn2(base_train)

In [15]:
for regras in regras_credit.rule_list:
    print(regras)

IF age>=34.9257164876908 THEN default=0 
IF income>=69478.3987640403 THEN default=1 
IF age>=34.915516287554105 THEN default=1 
IF age>=34.851817262359 THEN default=0 
IF age>=34.795262857340305 THEN default=1 
IF age>=34.7233597366139 THEN default=0 
IF age>=34.669146894011604 THEN default=1 
IF loan>=8066.69786524019 THEN default=1 
IF loan<=2507.64970973955 AND income>=20145.9885970689 THEN default=0 
IF income>=58132.4712652713 AND age>=22.939635145478 THEN default=0 
IF income>=58609.13148382679 AND age>=22.918212262913602 THEN default=1 
IF income>=58609.13148382679 THEN default=0 
IF loan>=5836.56338145928 AND age>=26.854012909811 THEN default=1 
IF age>=34.1016539284028 THEN default=0 
IF loan>=5836.56338145928 AND age>=26.7719294563867 THEN default=0 
IF loan>=5836.56338145928 AND age>=23.6371360039338 THEN default=1 
IF income<=32256.8615246564 AND loan>=3343.81635769923 THEN default=1 
IF loan>=5836.56338145928 AND loan>=6415.0862444378 THEN default=1 
IF income>=48552.84340

In [16]:
previsoes = Orange.evaluation.testing.TestOnTestData(base_train, base_test, [lambda testdata: regras_credit])

In [17]:
Orange.evaluation.CA(previsoes)

array([0.972])

## Classificador base - Majority Learner

### Base credit data

In [18]:
base_credit.domain

[income, age, loan | default]

In [19]:
majority = Orange.classification.MajorityLearner()

In [20]:
previsoes = Orange.evaluation.testing.TestOnTestData(base_credit, base_credit, [majority])

In [21]:
Orange.evaluation.CA(previsoes)

array([0.8585])

In [22]:
from collections import Counter
Counter(str(registro.get_class()) for registro in base_credit)

Counter({'0': 1717, '1': 283})

### Base census